# # Step 0: Auto-Install Dependencies (Colab Compatibility)
# Ensures `gymnasium` and `pygame` are installed when running in a fresh Google Colab runtime.

In [ ]:
import subprocess
import sys

def ensure_colab_packages():
    """Installs gymnasium[toy-text] quietly if not already present in the Colab environment."""
    try:
        import gymnasium  # noqa: F401
    except ImportError:
        print("Installing gymnasium[toy-text] for Google Colab...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gymnasium[toy-text]"])
        print("Installation complete!")

ensure_colab_packages()

In [ ]:
import random
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

# # Part 1: Environment & Training Hyperparameters
# Start with `IS_SLIPPERY = False` for a deterministic classroom demonstration, then switch to `True` to test stochastic ice physics.

In [ ]:
ENV_NAME = "FrozenLake-v1"
IS_SLIPPERY = True          # False = deterministic demo; True = slippery stochastic ice
LEARNING_RATE = 1e-3         # Adam optimizer step size (alpha)
GAMMA = 0.99                 # Discount factor for future rewards
BUFFER_CAPACITY = 10000      # Replay buffer capacity (N)
BATCH_SIZE = 64              # Mini-batch size per SGD update (B)
EPSILON_START = 1.0          # Initial exploration rate (100% random)
EPSILON_END = 0.01           # Minimum exploration rate (1% random)
EPSILON_DECAY = 0.995        # Multiplicative decay factor per episode
TARGET_UPDATE_FREQ = 20      # Sync target network every C = 20 episodes
NUM_EPISODES = 600           # Total training episodes
HIDDEN_DIM = 64              # Hidden layer width in the MLP
SEED = 42                    # Reproducibility seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

# Part 2: One-Hot State Encoding
# Maps discrete grid state $s \in \{0, \dots, 15\}$ to a unit basis vector $\mathbf{e}_s \in \{0, 1\}^{16}$ so the network does not treat tile IDs as continuous magnitudes.

In [ ]:
def one_hot_encode(state: int, num_states: int, dev: torch.device) -> torch.Tensor:
    """Converts an integer state index (0..15) into a one-hot float tensor of shape (16,)."""
    vec = torch.zeros(num_states, dtype=torch.float32, device=dev)
    vec[state] = 1.0
    return vec

# # Part 3: Deep Q-Network (DQN) Model
# A 3-layer Multi-Layer Perceptron (`16 -> 64 -> 64 -> 4`) that predicts $Q_\theta(s, a)$ for all 4 actions (`Left`, `Down`, `Right`, `Up`) in a single forward pass.

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)  # Linear output layer (unbounded Q-values)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# # Part 4: Experience Replay Buffer
# Stores transitions $(s_t, a_t, r_t, s_{t+1}, d_t)$ in an $O(1)$ circular `deque` and samples uniform random mini-batches to break sequential correlations.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state: int, action: int, reward: float, next_state: int, done: bool):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return states, actions, rewards, next_states, dones

    def __len__(self) -> int:
        return len(self.buffer)

# # Part 5: DQN Agent ($\epsilon$-Greedy Policy & Bellman Update)
# Implements `select_action()` for exploration vs. exploitation and `update()` for minimizing the Temporal-Difference MSE loss against the frozen target network $\theta^-$.

In [ ]:
class FrozenLakeDQNAgent:
    def __init__(self, num_states: int, num_actions: int, dev: torch.device):
        self.num_states = num_states
        self.num_actions = num_actions
        self.device = dev

        # Policy network (updated every step) and Target network (synced every C episodes)
        self.policy_net = DQN(num_states, num_actions, HIDDEN_DIM).to(dev)
        self.target_net = DQN(num_states, num_actions, HIDDEN_DIM).to(dev)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.loss_fn = nn.MSELoss()
        self.memory = ReplayBuffer(BUFFER_CAPACITY)
        self.epsilon = EPSILON_START

    def select_action(self, state: int, greedy: bool = False) -> int:
        if not greedy and random.random() < self.epsilon:
            return random.randrange(self.num_actions)

        with torch.no_grad():
            state_tensor = one_hot_encode(state, self.num_states, self.device).unsqueeze(0)
            q_values = self.policy_net(state_tensor)
            return int(torch.argmax(q_values, dim=1).item())

    def get_all_q_values(self) -> np.ndarray:
        """Returns the full (16, 4) Q-value table predicted by policy_net for visualization."""
        with torch.no_grad():
            eye = torch.eye(self.num_states, dtype=torch.float32, device=self.device)
            return self.policy_net(eye).cpu().numpy()

    def update(self) -> float:
        if len(self.memory) < BATCH_SIZE:
            return 0.0

        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)

        state_batch = torch.stack([one_hot_encode(s, self.num_states, self.device) for s in states])
        next_state_batch = torch.stack([one_hot_encode(s, self.num_states, self.device) for s in next_states])
        action_batch = torch.tensor(actions, dtype=torch.int64, device=self.device).unsqueeze(1)
        reward_batch = torch.tensor(rewards, dtype=torch.float32, device=self.device).unsqueeze(1)
        done_batch = torch.tensor(dones, dtype=torch.float32, device=self.device).unsqueeze(1)

        # Predicted Q(s, a) for the actions actually executed
        current_q = self.policy_net(state_batch).gather(1, action_batch)

        # Bellman target: y = r + gamma * max_{a'} Q_target(s', a') * (1 - done)
        with torch.no_grad():
            max_next_q = self.target_net(next_state_batch).max(dim=1, keepdim=True)[0]
            target_q = reward_batch + (GAMMA * max_next_q * (1.0 - done_batch))

        loss = self.loss_fn(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return float(loss.item())

    def sync_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)

# # Part 6: Training Loop & Colab Visualizations
# Trains the agent for 600 episodes and plots:
# 1. Rolling 50-episode success rate (%) and $\epsilon$ decay curve.
# 2. Learned $4 \times 4$ state-value heatmap $\max_a Q(s,a)$ overlaid with greedy policy arrows.

In [ ]:
def plot_colab_dashboard(rewards_history, epsilons_history, losses_history, agent):
    """Generates a 2-panel Matplotlib figure suitable for inline display in Google Colab."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Panel 1: Rolling 50-Episode Success Rate & Epsilon Decay
    ax1 = axes[0]
    window = 50
    rolling_success = [
        np.mean(rewards_history[max(0, i - window + 1):i + 1]) * 100
        for i in range(len(rewards_history))
    ]
    ax1.plot(rolling_success, color="#0f3460", linewidth=2, label="Success Rate (50-ep moving avg)")
    ax1.set_xlabel("Episode")
    ax1.set_ylabel("Success Rate (%)", color="#0f3460")
    ax1.set_ylim(-5, 105)
    ax1.grid(True, linestyle="--", alpha=0.4)

    ax1_twin = ax1.twinx()
    ax1_twin.plot(epsilons_history, color="#008094", linestyle="--", linewidth=1.8, label="Epsilon (Exploration)")
    ax1_twin.set_ylabel("Epsilon (ε)", color="#008094")
    ax1_twin.set_ylim(0, 1.05)
    ax1.set_title("DQN Training Progress on FrozenLake-v1", fontweight="bold")

    # Panel 2: 4x4 Learned Max Q-Value Heatmap & Greedy Policy Arrows
    ax2 = axes[1]
    q_table = agent.get_all_q_values()
    max_q_grid = np.max(q_table, axis=1).reshape(4, 4)
    best_actions = np.argmax(q_table, axis=1).reshape(4, 4)

    im = ax2.imshow(max_q_grid, cmap="YlGnBu", vmin=0.0, vmax=1.0)
    plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04, label="Max Predicted Q(s, a)")

    grid_labels = [
        ["S", "F", "F", "F"],
        ["F", "H", "F", "H"],
        ["F", "F", "F", "H"],
        ["H", "F", "F", "G"]
    ]
    arrow_map = {0: "←", 1: "↓", 2: "→", 3: "↑"}

    for r in range(4):
        for c in range(4):
            tile = grid_labels[r][c]
            val = max_q_grid[r, c]
            arrow = arrow_map[best_actions[r, c]]
            text_color = "white" if val > 0.6 else "black"
            if tile == "H":
                label_text = f"Hole (s={r*4+c})\n0.00"
            elif tile == "G":
                label_text = f"Goal (s=15)\n★"
            else:
                label_text = f"{tile} (s={r*4+c})\n{arrow}  {val:.2f}"
            ax2.text(c, r, label_text, ha="center", va="center", color=text_color, fontweight="bold", fontsize=10)

    ax2.set_xticks(range(4))
    ax2.set_yticks(range(4))
    ax2.set_title("Learned 4x4 State Values & Greedy Policy", fontweight="bold")

    plt.tight_layout()
    plt.savefig("frozen_lake_dqn_results.png", dpi=150)
    plt.show()

# # Part 7: Execute Training & Greedy Evaluation Rollout

In [ ]:
def main():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY)
    num_states = env.observation_space.n
    num_actions = env.action_space.n

    agent = FrozenLakeDQNAgent(num_states, num_actions, device)
    rewards_history, epsilons_history, losses_history = [], [], []

    print(f"\nStarting DQN Training on {ENV_NAME} (is_slippery={IS_SLIPPERY})...")
    for episode in range(1, NUM_EPISODES + 1):
        state, _ = env.reset(seed=SEED + episode)
        done = False
        episode_reward = 0.0
        ep_losses = []

        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.memory.push(state, action, reward, next_state, done)
            loss_val = agent.update()
            if loss_val > 0:
                ep_losses.append(loss_val)

            state = next_state
            episode_reward += reward

        agent.decay_epsilon()
        rewards_history.append(episode_reward)
        epsilons_history.append(agent.epsilon)
        losses_history.append(np.mean(ep_losses) if ep_losses else 0.0)

        if episode % TARGET_UPDATE_FREQ == 0:
            agent.sync_target_network()

        if episode % 50 == 0:
            recent_rate = np.mean(rewards_history[-50:]) * 100
            print(
                f"Episode {episode:4d}/{NUM_EPISODES} | "
                f"Success Rate (Last 50): {recent_rate:5.1f}% | "
                f"Epsilon: {agent.epsilon:.3f}"
            )

    env.close()

    # Display visual dashboard in Colab (and save PNG artifact)
    plot_colab_dashboard(rewards_history, epsilons_history, losses_history, agent)

    # Run a single greedy test rollout
    action_names = {0: "Left (←)", 1: "Down (↓)", 2: "Right (→)", 3: "Up (↑)"}
    eval_env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY, render_mode="ansi")
    state, _ = eval_env.reset(seed=SEED)
    done = False
    step_count = 0

    print("\nGreedy Evaluation Episode Trajectory:")
    print(eval_env.render())
    while not done and step_count < 20:
        action = agent.select_action(state, greedy=True)
        state, reward, terminated, truncated, _ = eval_env.step(action)
        done = terminated or truncated
        step_count += 1
        print(f"Step {step_count}: {action_names[action]} -> Tile {state}")
        if done:
            print("Outcome:", "Reached the Goal! 🎉" if reward > 0 else "Fell in a Hole! ❄️")
    eval_env.close()

In [ ]:
if __name__ == "__main__":
    main()